# Prerequisite: Jacobians, Hessians and Newton's method

**The challenge.** The lectures of this course use three words on almost every
page: *gradient*, *Jacobian*, *Hessian*. They are not advanced ideas. Each
answers one plain question about a function with several inputs: if I move the
inputs a little, what happens? This lecture builds them from single-variable
calculus and matrix multiplication, which is all it assumes, and it shows what
each one is *for*: the Jacobian turns a set of equations into a sequence of
linear solves, and the Hessian says whether a point is a minimum at all.

| Function | Its derivative | The question it answers |
|---|---|---|
| one number to one number | the slope $f'(x)$ | how fast does the output move? |
| many numbers to one number | the **gradient** $\nabla f$ | which way is uphill, and how steep? |
| many numbers to many numbers | the **Jacobian** $J$ | how does each output respond to each input? |
| many numbers to one number, again | the **Hessian** $H$ | how does the surface curve, and is this a bottom? |

## 1. The one idea: a derivative is a linear approximation

Single-variable calculus says that near $x$,

$$
f(x + h) \approx f(x) + f'(x)\,h .
$$

Everything in this lecture is that line with more inputs and outputs. The
derivative is whatever you multiply a small move by to get the change in the
output, and the error is smaller than the move: halve $h$ and the error of the
approximation falls by four.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

for h in (0.1, 0.05, 0.025):
    error = abs(np.exp(1 + h) - (np.exp(1) + np.exp(1) * h))
    print(f"h = {h:<6} error of the linear approximation to exp at 1: {error:.3e}")

## 2. Many inputs, one output: the gradient

Take a function of two inputs, a landscape with height $f(x, y)$. Freeze $y$ and
differentiate in $x$: that is the *partial derivative* $\partial f/\partial x$.
Stack the partial derivatives into a vector and you have the **gradient**:

$$
\nabla f = \begin{bmatrix} \partial f/\partial x \\ \partial f/\partial y \end{bmatrix},
\qquad
f(\mathbf x + \mathbf d) \approx f(\mathbf x) + \nabla f(\mathbf x) \cdot \mathbf d .
$$

The same line as before, with a dot product. That dot product says everything
about the gradient. The change is largest when $\mathbf d$ points along
$\nabla f$, so **the gradient points straight uphill** and its length is the
slope in that direction. The change is zero when $\mathbf d$ is perpendicular
to it, so **the gradient is perpendicular to the contour lines**. And at the
bottom of a valley no direction goes down to first order, so $\nabla f = 0$
there: that is the condition every optimization starts from.

In [ ]:
x, y = sp.symbols("x y", real=True)
f = x ** 2 / 2 + y ** 2 + x * y / 3
gradient = [sp.diff(f, s) for s in (x, y)]
print("grad f =", gradient)

f_num = sp.lambdify((x, y), f, "numpy")
g_num = sp.lambdify((x, y), gradient, "numpy")
point, direction = np.array([1.2, -0.7]), np.array([0.6, 0.8])
small = 1e-6
print("change along d, per unit step :", (f_num(*(point + small * direction)) - f_num(*point)) / small)
print("grad f . d                    :", np.dot(g_num(*point), direction))

In [ ]:
X, Y = np.meshgrid(np.linspace(-2.4, 2.4, 300), np.linspace(-1.8, 1.8, 300))
fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.contour(X, Y, f_num(X, Y), levels=12, colors=GRAY, linewidths=0.9)
for px, py in [(1.2, -0.7), (-1.5, 0.3), (0.4, 1.0), (-0.5, -1.0), (1.7, 0.7)]:
    gx, gy = g_num(px, py)
    ax.quiver(px, py, 0.22 * gx, 0.22 * gy, angles="xy", scale_units="xy", scale=1, color=BLUE,
              width=0.006, zorder=3)
    ax.plot(px, py, "o", color=BLUE, ms=5)
ax.plot(0, 0, "o", color=ORANGE, ms=9)
ax.annotate(r"the bottom: $\nabla f = 0$", (0, 0), xytext=(0.25, -0.2), color="k")
ax.set(xlabel="x", ylabel="y", aspect="equal")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

*Contours of $f$ with the gradient drawn at five points. Each arrow crosses its contour at a right angle and points uphill, and arrows are longer where the contours are closer together. The arrows shrink to nothing at the bottom.*

## 3. Many inputs, many outputs: the Jacobian

In 1841 Carl Gustav Jacob Jacobi published a long paper on what he called
*functional determinants*, in order to change variables in multiple integrals.
He wanted to know how much a small patch of area stretches under a mapping. The
answer is the determinant of a matrix of partial derivatives, and the matrix
now carries his name.

For a function $\mathbf F$ with $n$ inputs and $m$ outputs, the **Jacobian** is
the $m \times n$ matrix of every output's derivative with respect to every input:

$$
J_{ij} = \frac{\partial F_i}{\partial x_j},
\qquad
\mathbf F(\mathbf x + \mathbf d) \approx \mathbf F(\mathbf x) + J(\mathbf x)\,\mathbf d .
$$

The same line again, now with a matrix. There are two ways to read the matrix,
and both are used constantly:

- **a row** is the gradient of one output;
- **a column** is what one input does to all the outputs.

**A robot's Jacobian.** Take the arm of the later lectures: two links of lengths
$l_1$ and $l_2$, whose joint angles $(q_1, q_2)$ place the hand at $(p_x, p_y)$.

In [ ]:
q1, q2 = sp.symbols("q1 q2", real=True)
l1, l2 = 1.0, 0.8
hand = sp.Matrix([l1 * sp.cos(q1) + l2 * sp.cos(q1 + q2),
                  l1 * sp.sin(q1) + l2 * sp.sin(q1 + q2)])
J = hand.jacobian([q1, q2])
display(J)
print("det J =", sp.simplify(J.det()))

hand_num = sp.lambdify((q1, q2), list(hand), "numpy")
J_num = sp.lambdify((q1, q2), J, "numpy")

q = np.array([0.5, 1.6])
move = np.array([0.01, -0.02])          # a small change of the joint angles
print("the hand really moves by:", np.array(hand_num(*(q + move))) - np.array(hand_num(*q)))
print("J times the joint move  :", J_num(*q) @ move)

So the Jacobian does what a robot programmer needs: **it turns joint rates into
hand velocity**, $\dot{\mathbf p} = J(\mathbf q)\,\dot{\mathbf q}$, with the
columns saying how each joint alone moves the hand.

**What the determinant says.** A square $J$ maps a small square of inputs to a
small parallelogram of outputs, and $|\det J|$ is the ratio of their areas. When
the determinant is zero the parallelogram collapses to a line: there is a
direction the outputs cannot move in, however the inputs change. That is a
**singular** Jacobian. Here $\det J = l_1 l_2 \sin q_2$, zero when the elbow is
straight. Physically, an outstretched arm cannot move its hand along its own
length.

In [ ]:
def outline(center, half=0.12, n=40):
    t = np.linspace(-half, half, n)
    edge = [np.column_stack([t, 0 * t - half]), np.column_stack([0 * t + half, t]),
            np.column_stack([-t, 0 * t + half]), np.column_stack([0 * t - half, -t])]
    return np.array(center) + np.vstack(edge)

fig, (left, right) = plt.subplots(1, 2, figsize=(11.5, 5.0), constrained_layout=True)
for center, colour, label in [((0.5, 1.6), BLUE, r"elbow bent: $\det J$ is large"),
                              ((0.5, 0.13), ORANGE, r"elbow almost straight: $\det J$ is small")]:
    square = outline(center)
    left.fill(square[:, 0], square[:, 1], color=colour, alpha=0.35)
    left.plot(*square.T, color=colour, lw=1.5, label=label)
    image = np.column_stack(hand_num(square[:, 0], square[:, 1]))
    right.fill(image[:, 0], image[:, 1], color=colour, alpha=0.35)
    right.plot(*image.T, color=colour, lw=1.5)
    base = np.array(hand_num(*center))
    tangent = base + (J_num(*center) @ (square - np.array(center)).T).T
    right.plot(*tangent.T, color="k", lw=0.8, ls="--")
    elbow = [l1 * np.cos(center[0]), l1 * np.sin(center[0])]
    right.plot([0, elbow[0], base[0]], [0, elbow[1], base[1]], "-o", color=colour, lw=2.5, ms=4)
left.set(xlabel=r"joint angle $q_1$", ylabel=r"joint angle $q_2$", title="the inputs: a small square of joint angles")
left.set_xlim(0.2, 1.5)
left.legend(frameon=False, loc="center right")
right.set(xlabel=r"hand $p_x$", ylabel=r"hand $p_y$", aspect="equal", title="the outputs: where the hand goes")
for ax in (left, right):
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

*The same small square of joint angles at two poses, and where it sends the hand (the arm is drawn at the centre of each square). With the elbow bent the square becomes a roomy patch. With the elbow nearly straight it is squashed nearly flat along the arc the hand can reach: the hand can hardly move along the arm. The dashed outlines are the Jacobian's prediction, which is the linear approximation of the true image.*

## 4. Jacobians multiply: the chain rule

If one function feeds another, $\mathbf z = \mathbf G(\mathbf y)$ and
$\mathbf y = \mathbf F(\mathbf x)$, a small move goes through both linear
approximations in turn, so the Jacobians multiply:

$$
J_{\mathbf G \circ \mathbf F} = J_{\mathbf G}\, J_{\mathbf F} .
$$

This is the rule behind Lecture 8. An integrator takes many small steps, each a
function of the last, so the Jacobian of the whole trajectory, the **state
transition matrix** $\Phi$, is the product of the Jacobians of the steps. In the
limit of small steps that product becomes a differential equation,
$\dot\Phi = A(t)\,\Phi$, where $A = \partial F/\partial z$ is the Jacobian of the
dynamics. The check below takes a thousand Euler steps of a pendulum and
multiplies their Jacobians.

In [ ]:
def step(z, dt):
    return z + dt * np.array([z[1], -np.sin(z[0])])

def step_jacobian(z, dt):
    return np.eye(2) + dt * np.array([[0.0, 1.0], [-np.cos(z[0]), 0.0]])

def run(z, n=1000, dt=0.002):
    phi = np.eye(2)
    for _ in range(n):
        phi = step_jacobian(z, dt) @ phi
        z = step(z, dt)
    return z, phi

z0 = np.array([0.8, 0.3])
_, phi = run(z0)
small = 1e-6
differences = np.column_stack([(run(z0 + small * e)[0] - run(z0 - small * e)[0]) / (2 * small)
                               for e in np.eye(2)])
print("product of the step Jacobians:\n", phi)
print("largest gap to finite differences:", np.abs(phi - differences).max())

## 5. What the Jacobian does: Newton's method

Most of this course ends in the same place: a set of equations
$\mathbf R(\mathbf v) = \mathbf 0$ with as many equations as unknowns. Nothing
solves nonlinear equations directly, but linear ones are easy, and the Jacobian
is exactly the linear version. Replace $\mathbf R$ by its linear approximation
and set that to zero:

$$
\mathbf R(\mathbf v) + J(\mathbf v)\,\boldsymbol\delta = \mathbf 0
\quad\Longrightarrow\quad
J(\mathbf v)\,\boldsymbol\delta = -\mathbf R(\mathbf v),
\qquad
\mathbf v \leftarrow \mathbf v + \boldsymbol\delta .
$$

Isaac Newton worked a version of this on the cubic $x^3 - 2x - 5 = 0$ in a
manuscript of 1669, without using derivatives explicitly. Joseph Raphson
published a cleaner form in 1690, and Thomas Simpson gave it in terms of calculus
and extended it to two equations in 1740.

Near a solution the method is startlingly fast, because what it ignores is
second order in the error: **the number of correct digits roughly doubles each
step.** Here it is doing inverse kinematics, finding the joint angles that put
the hand at a target.

In [ ]:
def newton(target, q, jacobian, steps=40, stop=1e-13):
    history = []
    for _ in range(steps):
        residual = np.array(hand_num(*q)) - target
        history.append(np.linalg.norm(residual))
        if history[-1] < stop:
            break
        try:
            q = q - np.linalg.solve(jacobian(q), residual)
        except np.linalg.LinAlgError:
            break
    return q, history

target = np.array(hand_num(0.7, 1.1))
solved, good = newton(target, np.array([0.3, 0.6]), lambda q: J_num(*q))
print("joint angles found:", solved)
for i, e in enumerate(good):
    print(f"  step {i}: |R| = {e:.2e}")

**Two ways it goes wrong.** Both matter later in this course.

*A wrong Jacobian.* Newton's method needs an accurate Jacobian to be fast. An
approximate one still points roughly the right way, but the error then shrinks by
only a fixed factor per step, and a bad enough one need not converge at all. While the
six-joint arm was being built for this package, a bug made one second derivative
come out as exactly zero. The Jacobian of the shooting equations was wrong in a
few entries, and the solver stalled on problems it should have solved in seconds.
Comparing the Jacobian with finite differences, as in Section 4, found the bug.

*A singular Jacobian at the solution.* Send the hand to the very edge of its reach,
where the elbow is straight. The solution exists, but the Jacobian is singular
there, and the equations have a **double root**: like $x^2 = 0$, not $x = 0$. The
error in the elbow angle then only halves per step, as it does for Newton on
$x^2 = 0$, and the residual, which is quadratic in that error, falls by four. This is why
Lecture 14 and Lecture 19 replace conditions that have double roots at the
junctions of a boundary arc by equivalent ones with simple roots.

In [ ]:
def halved_entry(q):
    wrong = np.array(J_num(*q), dtype=float)
    wrong[0, 1] *= 0.5                    # one entry of the matrix is wrong
    return wrong

_, wrong = newton(target, np.array([0.3, 0.6]), halved_entry)
reach = np.array([l1 + l2, 0.0])          # the edge of the workspace: q2 = 0 is singular
_, edge = newton(reach, np.array([0.3, 0.6]), lambda q: J_num(*q))
print("steps to |R| < 1e-13: correct", len(good) - 1, " one wrong entry", len(wrong) - 1,
      " edge of reach", len(edge) - 1)

fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.semilogy(good, "-o", color=BLUE, lw=2, ms=6, label="correct Jacobian, regular solution")
ax.semilogy(wrong, "-s", color=ORANGE, lw=2, ms=6, label="one entry of the Jacobian wrong")
ax.semilogy(edge, "-^", color=GRAY, lw=2, ms=6, label="correct Jacobian, singular at the solution")
ax.set(xlabel="Newton step", ylabel=r"$\|\mathbf{R}\|$")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

*On a log scale, doubling digits is a curve that plunges, and a fixed factor per step is a straight line. With the correct Jacobian at a regular solution, the residual plunges in a few steps. A wrong entry or a singular solution each turn that into a straight line: both still converge here, but they take three to four times as many steps. Entries wrong by more can be far slower: a factor of ten instead of two still leaves $\|\mathbf{R}\|$ near $10^{-4}$ after 60 steps.*

## 6. The Hessian: how the surface curves

Otto Hesse, who had studied under Jacobi at Königsberg, used determinants of second
derivatives in the 1840s to find the inflection points of cubic curves. James
Joseph Sylvester named the determinant "the Hessian" in 1851, and the name passed
to the matrix.

The gradient gave a tilted plane. A plane has no bottom, so to tell a valley from a
ridge we need one more term. The **Hessian** is the matrix of second derivatives:

$$
H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j},
\qquad
f(\mathbf x + \mathbf d) \approx f(\mathbf x) + \nabla f \cdot \mathbf d
+ \tfrac12\, \mathbf d^\top H\, \mathbf d .
$$

Three facts are worth keeping.

- **The Hessian is the Jacobian of the gradient.** The gradient is a function with
  $n$ inputs and $n$ outputs, and differentiating it gives $H$.
- **It is symmetric**: the order of differentiation does not matter for a smooth
  function. So it has real eigenvalues, and perpendicular directions along which
  it acts as pure stretching.
- **Its eigenvalues are the curvatures** along those directions.

**What it does: it decides what a flat point is.** At a point where $\nabla f = 0$,
the term $\tfrac12 \mathbf d^\top H \mathbf d$ is all that is left, and the
eigenvalues read off the answer:

| Eigenvalues of $H$ | The point is | The matrix is called |
|---|---|---|
| all positive | a minimum: every direction curves up | positive definite |
| all negative | a maximum | negative definite |
| mixed signs | a **saddle**: up one way, down another | indefinite |
| some zero | undecided at this order: there is a flat direction | singular |

A zero gradient alone does not make a minimum. The last row matters in this course:
a flat direction means there is a whole family of equally good answers, and a
singular Hessian is how that shows up in the numbers (Lectures 18 and 19).

In [ ]:
surfaces = [("a bowl", x ** 2 + 2 * y ** 2), ("a saddle", x ** 2 - y ** 2), ("a flat valley", x ** 2)]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.9), constrained_layout=True)
for ax, (name, g) in zip(axes, surfaces):
    H = sp.hessian(g, (x, y))
    values = sorted(float(v) for v in H.eigenvals(multiple=True))
    print(f"{name:14s} H = {H.tolist()}  eigenvalues {values}")
    G = sp.lambdify((x, y), g, "numpy")
    A, B = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
    Z = G(A, B) * np.ones_like(A)
    ax.contour(A, B, Z, levels=np.linspace(Z.min() + 0.2, Z.max() - 0.2, 9),
               colors=GRAY, linewidths=0.9, linestyles="solid")
    if name == "a flat valley":
        ax.plot([0, 0], [-2, 2], color=ORANGE, lw=3)
    else:
        ax.plot(0, 0, "o", color=ORANGE, ms=8)
    ax.set(title=f"{name}: eigenvalues {values[0]:g}, {values[1]:g}", xlabel="x", ylabel="y", aspect="equal")
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

*Three surfaces whose gradient vanishes at the origin, told apart by their Hessians. In the valley on the right the whole orange line is equally low: the optimum is not unique, and one eigenvalue is zero.*

**What it does in a solver.** Finding a minimum means solving $\nabla f = \mathbf 0$,
and Newton's method on those equations uses the Jacobian of the gradient, which is
the Hessian: $H \boldsymbol\delta = -\nabla f$. The Hessian corrects for the shape of
the valley. Walking straight downhill, by contrast, zigzags across a narrow valley,
because downhill does not point along it.

In [ ]:
curvature = np.array([1.0, 100.0])        # a valley 100 times steeper one way than the other

def gradient_descent(v, tolerance=1e-6):
    best_step = 2 / (curvature.min() + curvature.max())
    steps = 0
    while np.linalg.norm(curvature * v) > tolerance:
        v = v - best_step * curvature * v
        steps += 1
    return steps

start = np.array([1.0, 1.0])
print("walking downhill, with the best fixed step:", gradient_descent(start), "steps")
after = start - np.linalg.solve(np.diag(curvature), curvature * start)
print("one Newton step lands at                  :", after)

## 7. Where they live in optimal control

All of this meets in the Hamiltonian $H(x, \lambda, u)$ (an unfortunate clash of
letters: $H$ here is the Hamiltonian, and its Hessian in $u$ is written $H_{uu}$).
Here is a pendulum, torqued from rest to rest with minimum effort, with each object
derived in view.

In [ ]:
from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.problem import OptimalControlProblem
from auto_optimal_control.shooting import SingleShooting

t, th, w, u = sp.symbols("t theta omega u", real=True)
T = sp.Symbol("T", positive=True)
pendulum = OptimalControlProblem(
    name="pendulum_effort", time=t, states=[th, w], controls=[u], params=[T],
    dynamics=[w, -sp.sin(th) + u], running_cost=u ** 2 / 2,
    initial_state=[0, 0], final_state={th: 1, w: 0}, final_time=T)
pmp = derive(pendulum)
lam_th, lam_w = pmp.costates

hamiltonian = u ** 2 / 2 + lam_th * w + lam_w * (-sp.sin(th) + u)
print("dH/du   =", sp.diff(hamiltonian, u), "  (a gradient: zero at the best control)")
print("d2H/du2 =", sp.diff(hamiltonian, u, 2), "  (a Hessian: positive, so that point is a minimum)")
print("u*      =", pmp.control_law[u])
print("costate equations, minus the gradient of H in x:", pmp.costate_dynamics)

state_and_costate = list(pmp.z)
A = sp.Matrix(pmp.dynamics).jacobian(state_and_costate)
print("A = dF/dz, the Jacobian of the extremal's dynamics:")
display(A)

Read that output against the table of this lecture:

| In the course | What it is |
|---|---|
| $\lambda' = -\partial H/\partial x$ (Lecture 2) | minus a **gradient** |
| $\lambda = \nabla V$, the costate as a value gradient (Lecture 5) | a **gradient** |
| $\partial H/\partial u = 0$ for the best control (Lecture 2) | a **gradient** set to zero |
| $H_{uu} \succeq 0$, the Legendre–Clebsch condition (Lecture 12) | a **Hessian** test: the point really is a minimum |
| $A = \partial F/\partial z$ (Lecture 8) | a **Jacobian**, whose lower-left block holds minus the Hessian $H_{xx}$ |
| $\Phi$, the state transition matrix (Lecture 8) | the **Jacobian** of the whole trajectory, by the chain rule |
| the shooting equations' $J$ (Lecture 9) | the **Jacobian** that Newton's method solves with |
| a singular $J$ for a non-unique optimum (Lectures 18, 19) | a flat direction |

The package compiles $A$ into C. It should agree with finite differences of the
compiled dynamics, and it does. With the Jacobian of the shooting equations, a bare
Newton loop of the kind in Section 5 solves the optimal control problem:

In [ ]:
k = build_kernels(pmp)
params = [2.0]                            # T
z = np.array([0.4, 0.2, -0.3, 0.5])
small = 1e-6
differences = np.column_stack([(k.rhs(0.5, z + small * e, params) - k.rhs(0.5, z - small * e, params)) / (2 * small)
                               for e in np.eye(4)])
print("compiled A vs finite differences, largest gap:", np.abs(k.jacobian(0.5, z, params) - differences).max())

shoot = SingleShooting(k, params, rtol=1e-12, atol=1e-12)
v = shoot.pack(np.zeros(2), np.zeros(k.N_MULTIPLIERS))   # the unknowns: initial costates, multipliers
for i in range(8):
    R, jac = shoot.residual_jacobian(v)
    print(f"  step {i}: |R| = {np.linalg.norm(R):.2e}")
    if np.linalg.norm(R) < 1e-12:
        break
    v = v - np.linalg.solve(jac, R)
print("initial costates:", shoot.unpack(v)[1])

## 8. How a computer gets them

There are three ways to get a derivative, and each has a place in this package.

**Finite differences.** Evaluate the function twice and divide. It takes no
thought, and it is the standard way to *check* a derivative. But it costs an
evaluation or two per input, and its accuracy is limited: a large step has
truncation error, a small one loses digits to round-off, and no step is good. The
central difference is much better than the one-sided one, and neither approaches
the sixteen digits of the arithmetic.

In [ ]:
steps = np.logspace(-14, -1, 80)
at = 1.0
forward = np.abs((np.sin(at + steps) - np.sin(at)) / steps - np.cos(at))
central = np.abs((np.sin(at + steps) - np.sin(at - steps)) / (2 * steps) - np.cos(at))
print(f"smallest error: one-sided {forward.min():.1e} at h = {steps[forward.argmin()]:.0e}; "
      f"central {central.min():.1e} at h = {steps[central.argmin()]:.0e}")
fig, ax = plt.subplots(figsize=(7.6, 4.0))
ax.loglog(steps, forward, color=BLUE, lw=2, label="one-sided difference")
ax.loglog(steps, central, color=ORANGE, lw=2, label="central difference")
ax.set(xlabel="step size h", ylabel="error in the derivative of sin at 1")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

*The error of a finite difference against its step. To the right of each dip, truncation error falls as the step shrinks. To the left, round-off grows. The minimum is where the two meet, and it is far from machine precision.*

**Symbolic differentiation.** SymPy applies the rules of calculus to a formula, and
the result is exact. This is how most of the package's kernels start. Its weakness
is that formulas written out can be enormous. **Automatic differentiation** avoids
that by working on the program instead of the formula. A program is a list of small
steps, each with a known derivative, and the chain rule of Section 4 can be applied
to the list. R. E. Wengert described the forward form in 1964, and Seppo Linnainmaa
the reverse form in his 1970 master's thesis. The package's tape does this for large
expressions, such as the torques of a robot with six joints:

In [ ]:
from auto_optimal_control.robot import SIX_JOINT_LINKS, joint_torques
from auto_optimal_control.tape import Tape

qs, ws, accels = (sp.symbols(f"{s}1:7", real=True) for s in ("q", "w", "a"))
torques = joint_torques(SIX_JOINT_LINKS, qs, ws, accels, gravity=sp.Rational(981, 100))
print("operations in the first torque, written out:", sp.count_ops(torques[0]))

tape = Tape()
roots = [tape.from_sympy(e) for e in torques]
print("steps for all six torques, shared          :", len(tape.reachable(roots)))

names = [s.name for s in (*qs, *ws, *accels)]
total = roots[0]
for r in roots[1:]:
    total = tape.add(total, r)                # one scalar: the sum of the torques
before = len(tape.reachable([total]))
reverse = tape.gradient(total, names)
after_reverse = len(tape.reachable(reverse))
forward = [tape.diff(total, n) for n in names]
after_forward = len(tape.reachable(forward))
print(f"its 18-entry gradient: reverse {after_reverse} steps, forward {after_forward} steps, "
      f"against {before} for the value")
print("the two agree:", np.allclose(*[tape.evaluate(g, dict(zip(names, np.linspace(0.1, 1.8, 18))))
                                      for g in (reverse, forward)]))

**Forward or reverse?** Forward mode carries the derivative of every step with respect
to one input, so it gives one *column* of the Jacobian per pass. Reverse mode runs
through the program backwards from one output, so it gives one *row* per pass.

| | One pass gives | Cheapest when |
|---|---|---|
| Forward mode | one column of $J$ | few inputs, many outputs |
| Reverse mode | one row of $J$ | many inputs, few outputs: a gradient |

A Hessian combines them: reverse mode for the gradient, then forward mode through
that gradient, one column at a time. Lecture 8's state transition matrix is forward
mode in continuous time, with one column of $\Phi$ per initial condition. The
costates themselves are reverse mode: they carry the sensitivity of a single cost
backwards along the trajectory. That is one way to read what the whole indirect
method is doing.

## 9. Summary

| Object | Size, for $n$ inputs and $m$ outputs | What it does |
|---|---|---|
| gradient $\nabla f$ | $n$ | points uphill; zero at a flat point |
| Jacobian $J$ | $m \times n$ | maps small moves of the inputs to moves of the outputs; its singularity marks a direction lost |
| determinant of $J$ (square) | one number | how much it stretches area or volume; zero when singular |
| Hessian $H$ | $n \times n$, symmetric | the curvature; its eigenvalues decide minimum, maximum, saddle or flat direction |
| Newton's method | solves $J\boldsymbol\delta = -\mathbf R$ | doubles the correct digits near a regular solution |

If you can answer these, you have what the course assumes:

1. Why is the gradient perpendicular to a contour line?
2. In the arm's Jacobian, what does the second column mean physically?
3. What do the singular elbow pose and the equation $x^2 = 0$ have in common?
4. Why does a zero gradient not settle whether a point is a minimum, and what does?
5. You need the gradient of one cost with respect to a thousand inputs. Forward or
   reverse mode?

Lecture 1 begins the course proper.